In [1]:
import torch
from transformers import AutoProcessor, LlavaForConditionalGeneration, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

model_id = "llava-hf/llava-1.5-7b-hf"

processor = AutoProcessor.from_pretrained(model_id)
print("Processor loaded successfully!")

model = LlavaForConditionalGeneration.from_pretrained(
    model_id,
    # torch_dtype=torch.float16,
    quantization_config=bnb_config,
    device_map="auto",
    low_cpu_mem_usage=True,
    attn_implementation="eager",   # important
)
print("Model loaded successfully!")

/venv/vlm-llava/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Processor loaded successfully!


Loading weights:   1%|          | 4/686 [00:00<00:01, 529.83it/s, Materializing param=model.language_model.layers.0.mlp.down_proj.weight]/venv/vlm-llava/lib/python3.12/site-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading weights: 100%|██████████| 686/686 [00:06<00:00, 114.02it/s, Materializing param=model.vision_tower.vision_model.pre_layrnorm.weight]                        


Model loaded successfully!


In [2]:
from PIL import Image

def prepare_input(processor, img_path: str, prompt: str):
    image = Image.open(img_path).convert("RGB")

    def create_messages(img):
        return [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": img},
                    {"type": "text", "text": prompt},
                ],
            },
        ]

    inputs = processor.apply_chat_template(
        create_messages(image), 
        add_generation_prompt=True, 
        tokenize=True, 
        return_dict=True, 
        return_tensors="pt"
    )

    return inputs

In [3]:
def move_inputs_to_device(inputs, model):
    """
    Move processor outputs to the model device.

    Keeps integer tensors such as input_ids unchanged.
    Casts floating tensors such as pixel_values to model dtype.
    """
    device = next(model.parameters()).device
    model_dtype = getattr(model, "dtype", torch.float16)

    moved = {}

    for k, v in inputs.items():
        if torch.is_tensor(v):
            if torch.is_floating_point(v):
                moved[k] = v.to(device=device, dtype=model_dtype)
            else:
                moved[k] = v.to(device=device)
        else:
            moved[k] = v

    return moved

In [4]:
def sample_next_token(logits, temperature=0.0):
    """
    logits: [batch, vocab]
    return: [batch, 1]
    """
    if temperature is None or temperature <= 0:
        return torch.argmax(logits, dim=-1, keepdim=True)

    probs = torch.softmax(logits / temperature, dim=-1)
    return torch.multinomial(probs, num_samples=1)

In [5]:
def get_image_token_id(model, tokenizer):
    """
    Get LLaVA image token id.
    """
    image_token_id = getattr(model.config, "image_token_index", None)

    if image_token_id is None:
        image_token_id = tokenizer.convert_tokens_to_ids("<image>")

    return image_token_id

In [6]:
def resolve_image_token_indices(
    input_ids,
    token_attn_key_len,
    current_step,
    model,
    tokenizer,
):
    """
    Resolve image-token positions in the attention key dimension.

    This handles two cases:

    Case 1:
        input_ids already contains many expanded image tokens.

    Case 2:
        input_ids contains one <image> placeholder, and LLaVA internally
        expands it into many visual patch tokens.

    Assumption:
        batch size = 1, single image input.
    """
    image_token_id = get_image_token_id(model, tokenizer)

    raw_img_positions = (
        input_ids[0] == image_token_id
    ).nonzero(as_tuple=False).flatten()

    if len(raw_img_positions) == 0:
        return torch.empty(0, dtype=torch.long)

    raw_prompt_len = input_ids.shape[1]

    # During decoding:
    # key_len = expanded_prompt_len + number_of_generated_tokens_seen
    #
    # At step 0, after feeding the first generated token:
    # key_len = expanded_prompt_len + 1
    expanded_prompt_len = token_attn_key_len - (current_step + 1)

    # Case 1: image tokens are already expanded in input_ids.
    if len(raw_img_positions) > 1:
        return raw_img_positions.cpu()

    # Case 2: one placeholder expanded internally.
    placeholder_pos = int(raw_img_positions[0].item())

    # One placeholder token is replaced by num_image_tokens visual tokens.
    num_image_tokens = expanded_prompt_len - (raw_prompt_len - 1)

    if num_image_tokens <= 0:
        return torch.empty(0, dtype=torch.long)

    image_start = placeholder_pos
    image_end = placeholder_pos + num_image_tokens

    return torch.arange(image_start, image_end, dtype=torch.long)


In [7]:
def is_sentence_end_token(token_id, tokenizer, eos_token_id=None):
    """
    Check whether a token is sentence-ending or EOS.
    """
    token_id = int(token_id)

    if eos_token_id is not None and token_id == eos_token_id:
        return True

    # Original LLaMA/LLaVA dot ids commonly seen in older code.
    if token_id in {869, 29889}:
        return True

    token_text = tokenizer.decode(
        [token_id],
        skip_special_tokens=False,
    )

    stripped = token_text.strip()

    if stripped in {".", "!", "?"}:
        return True

    return False

In [8]:
def build_candidate_record(token_ids, probs, tokenizer):
    """
    Convert candidate token ids/probs into readable records.
    """
    records = []

    for token_id, prob in zip(token_ids, probs):
        token_id = int(token_id)
        prob = float(prob)

        records.append(
            {
                "token_id": token_id,
                "token_text": tokenizer.decode(
                    [token_id],
                    skip_special_tokens=False,
                ),
                "prob": prob,
            }
        )

    return records

In [9]:
def select_token_from_logits(logits, temperature=0.0):
    """
    Select next token from logits.

    temperature <= 0:
        greedy decoding

    temperature > 0:
        multinomial sampling
    """
    if temperature is None or temperature <= 0:
        return torch.argmax(logits, dim=-1, keepdim=True)

    probs = torch.softmax(logits / temperature, dim=-1)
    return torch.multinomial(probs, num_samples=1)

In [10]:
import re
import torch


def generate_sentence_with_attn(
    model,
    processor,
    inputs,
    max_new_tokens=64,
    min_new_tokens=3,
    temperature=0.0,
    stop_at_sentence_end=True,

    # Attention options
    selected_layers=None,
    keep_attn_on_cpu=True,
    return_full_attn=False,

    # Uncertainty/checkpoint options
    top_k=5,
    accumulate_prob=0.5,
    enable_uncertainty_check=True,
    checkpoint_once=True,
    stop_if_sentence_end_in_candidates=True,

    debug=False,
):
    """
    Generate a sentence from prepared LLaVA-HF inputs and collect attention maps
    at each decoding step.

    Expected input:
        inputs = prepare_input(processor, img_path, prompt)

    Important:
        model should be loaded with attn_implementation="eager"

    Returns:
        {
            "generated_text": str,
            "generated_ids": list[int],

            "confidence_path": "certainty" | "uncertainty",
            "has_uncertainty": bool,
            "first_uncertain_step": int or None,
            "uncertain_steps": list[int],

            "stop_reason": str,

            "steps": [
                {
                    "step": int,
                    "token_id": int,
                    "token_text": str,
                    "max_prob": float,
                    "selected_token_prob": float,
                    "is_low_confidence": bool,
                    "attn_by_layer": dict or None,
                    "image_attn_by_layer": {
                        layer_id: Tensor[num_heads, num_image_tokens]
                    }
                }
            ],

            "image_token_indices": Tensor[num_image_tokens],

            "checkpoint_input_ids": Tensor or None,
            "checkpoint_generated_ids": Tensor or None,
            "checkpoint_text": str or None,
            "candidates": list[dict] or None,
            "candidate_records": list[dict],
        }
    """

    def dprint(*args):
        if debug:
            print(*args)

    model.eval()

    tokenizer = processor.tokenizer
    device = next(model.parameters()).device

    inputs = move_inputs_to_device(inputs, model)
    input_ids = inputs["input_ids"]

    eos_token_id = tokenizer.eos_token_id
    if eos_token_id is None:
        eos_token_id = getattr(model.config, "eos_token_id", None)

    # -------------------------------------------------------
    # 1. Prefill pass: prompt + image.
    # -------------------------------------------------------
    prefill_outputs = model(
        **inputs,
        use_cache=True,
        output_attentions=False,
        return_dict=True,
    )

    past_key_values = prefill_outputs.past_key_values

    # Logits for the first generated token.
    logits_for_next = prefill_outputs.logits[:, -1, :]

    # -------------------------------------------------------
    # 2. Storage.
    # -------------------------------------------------------
    generated_ids = []
    steps = []

    image_token_indices = None

    checkpointed = False
    checkpoint_input_ids = None
    checkpoint_generated_ids = None
    checkpoint_text = None

    candidates = None
    candidate_records = []

    # New path-level confidence tracking
    has_uncertainty = False
    first_uncertain_step = None
    uncertain_steps = []

    # New general stop reason
    stop_reason = None

    # -------------------------------------------------------
    # 3. Decoding loop.
    # -------------------------------------------------------
    for step in range(max_new_tokens):
        probs_for_next = torch.softmax(logits_for_next, dim=-1)
        max_prob = float(torch.max(probs_for_next).item())

        is_low_confidence = (enable_uncertainty_check and max_prob < accumulate_prob)

        if is_low_confidence:
            has_uncertainty = True
            uncertain_steps.append(step)

            if first_uncertain_step is None:
                first_uncertain_step = step

        force_stop_token = None

        # ---------------------------------------------------
        # 3.1 Uncertainty path logic.
        # ---------------------------------------------------
        should_store_uncertainty = (is_low_confidence and (not checkpoint_once or not checkpointed))

        if should_store_uncertainty:
            top_k_probs, top_k_indices = torch.topk(probs_for_next, k=top_k, dim=-1)

            cumsum_probs = torch.cumsum(top_k_probs, dim=-1)

            # Keep the smallest prefix of top-k candidates whose cumulative
            # probability reaches accumulate_prob.
            cumsum_ids = (cumsum_probs >= accumulate_prob).nonzero(as_tuple=True)[1]

            if len(cumsum_ids) > 0:
                min_k = int(cumsum_ids[0].item()) + 1
            else:
                min_k = top_k

            selected_candidate_ids = (top_k_indices[0, :min_k].detach().cpu().tolist())

            selected_candidate_probs = (top_k_probs[0, :min_k].detach().cpu().tolist())

            current_candidate_record = {
                "step": step,
                "max_prob": max_prob,
                "threshold": accumulate_prob,
                "candidates": build_candidate_record(
                    selected_candidate_ids,
                    selected_candidate_probs,
                    tokenizer,
                ),
            }

            candidate_records.append(current_candidate_record)

            if candidates is None:
                candidates = current_candidate_record["candidates"]

            # Store checkpoint before appending the uncertain token.
            if not checkpointed:
                checkpointed = True

                if len(generated_ids) > 0:
                    gen_tensor = torch.tensor([generated_ids], dtype=input_ids.dtype,device=input_ids.device)

                    checkpoint_input_ids = torch.cat([input_ids, gen_tensor], dim=-1)

                    checkpoint_generated_ids = gen_tensor.clone()
                else:
                    checkpoint_input_ids = input_ids.clone()

                    checkpoint_generated_ids = torch.empty((1, 0),dtype=input_ids.dtype,device=input_ids.device,)

                checkpoint_text = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

            dprint(f"[uncertainty path] step={step}, max_prob={max_prob:.4f}")
            dprint("candidates:", current_candidate_record["candidates"])

            # Stop if sentence-end or EOS is inside candidate set.
            if stop_if_sentence_end_in_candidates:
                for cand_id in selected_candidate_ids:
                    if is_sentence_end_token(cand_id, tokenizer, eos_token_id=eos_token_id):
                        force_stop_token = int(cand_id)
                        stop_reason = "sentence_end_or_eos_in_candidates"
                        break

        # ---------------------------------------------------
        # 3.2 Choose next token.
        # ---------------------------------------------------
        if force_stop_token is not None:
            next_token = torch.tensor([[force_stop_token]], dtype=torch.long, device=device)
        else:
            next_token = select_token_from_logits(logits_for_next, temperature=temperature)

        token_id = int(next_token[0, 0].item())

        selected_token_prob = float(probs_for_next[0, token_id].item())

        token_text = tokenizer.decode([token_id],skip_special_tokens=False)

        # ---------------------------------------------------
        # 3.3 Feed chosen token back into model.
        #
        # This gives attention OF the generated token.
        # ---------------------------------------------------
        step_outputs = model(
            input_ids=next_token,
            past_key_values=past_key_values,
            use_cache=True,
            output_attentions=True,
            return_dict=True,
        )

        generated_ids.append(token_id)

        # ---------------------------------------------------
        # 3.5 PHG-friendly attention extraction.
        # ---------------------------------------------------
        num_layers = len(step_outputs.attentions)

        if selected_layers is None:
            layer_ids = list(range(num_layers))
        else:
            layer_ids = []
            for layer_id in selected_layers:
                if layer_id < 0:
                    layer_id = num_layers + layer_id
                layer_ids.append(layer_id)

        attn_by_layer = {}
        image_attn_by_layer = {}

        for layer_id in layer_ids:
            attn = step_outputs.attentions[layer_id]

            # attn shape:
            #   [batch, num_heads, query_len, key_len]
            #
            # During cached decoding:
            #   query_len = 1
            #
            # token_attn shape:
            #   [num_heads, key_len]
            token_attn = attn[0, :, -1, :].detach()

            # Resolve image token span once.
            if image_token_indices is None:
                image_token_indices = resolve_image_token_indices(
                    input_ids=input_ids,
                    token_attn_key_len=token_attn.shape[-1],
                    current_step=step,
                    model=model,
                    tokenizer=tokenizer,
                )

            valid_img_idx = image_token_indices[
                image_token_indices < token_attn.shape[-1]
            ]

            if keep_attn_on_cpu:
                token_attn_for_store = token_attn.float().cpu()
            else:
                token_attn_for_store = token_attn

            if return_full_attn:
                attn_by_layer[layer_id] = token_attn_for_store

            if len(valid_img_idx) > 0:
                if keep_attn_on_cpu:
                    img_idx = valid_img_idx.cpu()
                    image_attn = token_attn_for_store[:, img_idx]
                else:
                    img_idx = valid_img_idx.to(token_attn.device)
                    image_attn = token_attn[:, img_idx]

                image_attn_by_layer[layer_id] = image_attn

        steps.append(
            {
                "step": step,
                "token_id": token_id,
                "token_text": token_text,

                # Keep both:
                # max_prob = model confidence in its best candidate
                # selected_token_prob = probability of the actual selected token
                "max_prob": max_prob,
                "selected_token_prob": selected_token_prob,

                "is_low_confidence": is_low_confidence,
                "attn_by_layer": attn_by_layer if return_full_attn else None,
                "image_attn_by_layer": image_attn_by_layer,
            }
        )

        # ---------------------------------------------------
        # 3.6 Update cache and logits for next step.
        # ---------------------------------------------------
        past_key_values = step_outputs.past_key_values
        logits_for_next = step_outputs.logits[:, -1, :]

        # ---------------------------------------------------
        # 3.7 Stop conditions.
        # ---------------------------------------------------
        if stop_reason is not None:
            break

        if eos_token_id is not None and token_id == eos_token_id:
            stop_reason = "eos_generated"
            break

        decoded_text = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

        if (
            stop_at_sentence_end
            and len(generated_ids) >= min_new_tokens
            and re.search(r"[.!?]\s*$", decoded_text)
        ):
            stop_reason = "sentence_end_generated"
            break

    # -------------------------------------------------------
    # 4. Final path decision.
    # -------------------------------------------------------
    generated_text = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

    if image_token_indices is None:
        image_token_indices = torch.empty(0, dtype=torch.long)

    if stop_reason is None:
        stop_reason = "max_new_tokens_reached"

    if has_uncertainty:
        confidence_path = "uncertainty"
    else:
        confidence_path = "certainty"

    # -------------------------------------------------------
    # 5. Return.
    # -------------------------------------------------------
    return {
        "generated_text": generated_text,
        "generated_ids": generated_ids,

        # Sentence-level confidence path
        "confidence_path": confidence_path,
        "has_uncertainty": has_uncertainty,
        "first_uncertain_step": first_uncertain_step,
        "uncertain_steps": uncertain_steps,

        # General stopping reason
        "stop_reason": stop_reason,

        # PHG-ready outputs
        "steps": steps,
        "image_token_indices": image_token_indices.cpu(),

        # Checkpoint/candidate outputs
        "checkpoint_input_ids": (
            checkpoint_input_ids.detach().cpu()
            if checkpoint_input_ids is not None
            else None
        ),
        "checkpoint_generated_ids": (
            checkpoint_generated_ids.detach().cpu()
            if checkpoint_generated_ids is not None
            else None
        ),
        "checkpoint_text": checkpoint_text,
        "candidates": candidates,
        "candidate_records": candidate_records,
    }

In [12]:
from PIL import Image

inputs = prepare_input(
    processor=processor,
    img_path="testimg.jpg",
    prompt="Describe this image."
)

result = generate_sentence_with_attn(
    model=model,
    processor=processor,
    inputs=inputs,
    max_new_tokens=32,
    selected_layers=[-1],
    top_k=5,
    accumulate_prob=0.5,
    enable_uncertainty_check=True,
    checkpoint_once=True,
    stop_if_sentence_end_in_candidates=True,
    return_full_attn=False,
    debug=True,
)

[uncertainty path] step=10, max_prob=0.1772
candidates: [{'token_id': 13587, 'token_text': 'holding', 'prob': 0.17724609375}, {'token_id': 926, 'token_text': 'pos', 'prob': 0.0948486328125}, {'token_id': 22047, 'token_text': 'surrounded', 'prob': 0.08636474609375}, {'token_id': 411, 'token_text': 'with', 'prob': 0.0579833984375}, {'token_id': 22314, 'token_text': 'proud', 'prob': 0.042755126953125}]


In [14]:
print(result["generated_text"])
print(result["stop_reason"])
print(result["checkpoint_text"])
print(result["candidates"])

The image features a man standing in a kitchen, holding a frying pan in his hand.
sentence_end_generated
The image features a man standing in a kitchen,
[{'token_id': 13587, 'token_text': 'holding', 'prob': 0.17724609375}, {'token_id': 926, 'token_text': 'pos', 'prob': 0.0948486328125}, {'token_id': 22047, 'token_text': 'surrounded', 'prob': 0.08636474609375}, {'token_id': 411, 'token_text': 'with', 'prob': 0.0579833984375}, {'token_id': 22314, 'token_text': 'proud', 'prob': 0.042755126953125}]
